### Extracting Trope informations from TEI XML files and adding to metadata files
after this need to add reprint information

In [5]:
import os
import pandas as pd
import xml.etree.ElementTree as ET

In [ ]:
# ADDING TROPES

# ---------- CONFIG ----------
METADATA_PATH = "images_with_article_metadata.csv"  # <-- your current metadata sheet
OUTPUT_PATH = "metadata_with_tropes.csv"
OBJECTS_DIR = "objects"         # directory that contains the article folders

# mapping from n="…" to trope label
TROPE_MAP = {
    "1": "Indian Queen",
    "2": "Captivity",
    "3": "Noble Savage",
    "4": "Discovery",
    "5": "Wonder",
    "6": "Vanishing Indians",
    "7": "Origins",
    "8": "Girl Crusoe",
    "9": "Jump Overboard",
    "10": "Signs Easily Interpreted",
    "11": "Indians as Savage",
    "12": "No Boat Available",
    "13": "Anti-Russian Sentiment",
    "14": "Wild Child",
    "15": "Lazy Indian",
}

# ---------- FUNCTIONS ----------

def extract_tropes_from_xml(xml_path):
    """
    Given a path to a TEI XML file, return a semicolon-separated string of trope labels.
    - Looks for <span type="trope" n="X">
    - Maps X via TROPE_MAP
    - Deduplicates and returns labels in first-seen order
    """
    if not os.path.exists(xml_path):
        return ""

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        print(f"WARNING: Could not parse XML: {xml_path}")
        return ""

    labels_in_order = []

    # Iterate over all elements; handle namespaces by just using .endswith("span")
    for el in root.iter():
        if not el.tag.endswith("span"):
            continue
        if el.get("type") != "trope":
            continue

        n_val = el.get("n")
        if n_val is None:
            continue

        label = TROPE_MAP.get(n_val)
        if label is None:
            # Unknown trope number; you can log or ignore
            print(f"WARNING: Unknown trope number n='{n_val}' in {xml_path}")
            continue

        # Deduplicate while preserving order
        if label not in labels_in_order:
            labels_in_order.append(label)

    if not labels_in_order:
        return ""

    return "; ".join(labels_in_order)


# ---------- MAIN WORKFLOW ----------

def main():
    # 1. Load metadata
    starter_metadata_df = pd.read_csv(METADATA_PATH)

    if "article_id" not in starter_metadata_df.columns:
        raise ValueError("metadata CSV must contain an 'article_id' column")

    # Ensure 'tropes' column exists
    if "tropes" not in starter_metadata_df.columns:
        starter_metadata_df["tropes"] = ""

    # 2. For each unique article_id, extract tropes from the TEI file
    article_ids = starter_metadata_df["article_id"].dropna().unique()

    trope_by_article = {}

    for article_id in article_ids:
        # article_id might be float/number; make sure it's a string
        article_id_str = str(article_id)

        # Path like: objects/AlbanyEveningJournal1853/m_AlbanyEveningJournal1853_TEI.xml
        xml_path = os.path.join(
            OBJECTS_DIR,
            article_id_str,
            f"m_{article_id_str}_TEI.xml"
        )

        tropes = extract_tropes_from_xml(xml_path)
        trope_by_article[article_id_str] = tropes

    # 3. Fill the 'tropes' column for each row based on article_id
    def lookup_tropes(row):
        article_id_str = str(row["article_id"])
        return trope_by_article.get(article_id_str, "")

    starter_metadata_df["tropes"] = starter_metadata_df.apply(lookup_tropes, axis=1)

    # 4. Save output
    starter_metadata_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved updated metadata with tropes to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


Saved updated metadata with tropes to: metadata_with_tropes.csv


In [ ]:


def read_transcript_file(path):
    """
    Read transcript text from a file path.

    Strategy:
    - Read bytes
    - Try multiple encodings: utf-8, utf-8-sig, utf-16-le, utf-16-be, latin-1
    - For each successful decode, compute how "ASCII-ish" it looks
    - Choose the decoding with the highest ASCII ratio
    - Collapse whitespace so it fits nicely into one CSV cell
    """
    if not os.path.exists(path):
        print(f"WARNING: Transcript file not found: {path}")
        return ""

    try:
        with open(path, "rb") as f:
            data = f.read()
    except Exception as e:
        print(f"WARNING: Could not open transcript {path}: {e}")
        return ""

    encodings_to_try = [
        "utf-8",
        "utf-8-sig",
        "utf-16-le",
        "utf-16-be",
        "latin-1",
    ]

    best_text = None
    best_score = -1
    last_error = None

    for enc in encodings_to_try:
        try:
            text = data.decode(enc)
        except UnicodeError as e:
            last_error = e
            continue

        # Compute how much this looks like plain ASCII-ish English
        # (ignore whitespace when scoring)
        non_space_chars = [ch for ch in text if not ch.isspace()]
        if not non_space_chars:
            score = 0
        else:
            ascii_chars = [
                ch for ch in non_space_chars
                if 0x20 <= ord(ch) <= 0x7E
            ]
            score = len(ascii_chars) / len(non_space_chars)

        # Keep the best scoring decoding
        if score > best_score:
            best_score = score
            best_text = text

    if best_text is None:
        print(f"WARNING: Could not decode transcript {path}: {last_error}")
        return ""

    # Collapse whitespace/newlines so it plays nicely in CSV cells
    cleaned = " ".join(best_text.split())
    return cleaned


In [10]:


# ---------- CONFIG ----------
METADATA_PATH = "metadata_with_tropes.csv"              # input CSV
OUTPUT_PATH = "metadata_with_tropes_transcripts.csv"    # output CSV
OBJECTS_DIR = "objects"                                 # directory containing article folders


def read_transcript_file(path):
    """
    Read transcript text from a file path.

    Strategy:
    - Read bytes
    - Try multiple encodings: utf-8, utf-8-sig, utf-16-le, utf-16-be, latin-1
    - For each successful decode, compute how "ASCII-ish" it looks
    - Choose the decoding with the highest ASCII ratio
    - Collapse whitespace so it fits nicely into one CSV cell
    """
    if not os.path.exists(path):
        print(f"WARNING: Transcript file not found: {path}")
        return ""

    try:
        with open(path, "rb") as f:
            data = f.read()
    except Exception as e:
        print(f"WARNING: Could not open transcript {path}: {e}")
        return ""

    encodings_to_try = [
        "utf-8",
        "utf-8-sig",
        "utf-16-le",
        "utf-16-be",
        "latin-1",
    ]

    best_text = None
    best_score = -1
    last_error = None

    for enc in encodings_to_try:
        try:
            text = data.decode(enc)
        except UnicodeError as e:
            last_error = e
            continue

        # Compute how much this looks like plain ASCII-ish English
        # (ignore whitespace when scoring)
        non_space_chars = [ch for ch in text if not ch.isspace()]
        if not non_space_chars:
            score = 0
        else:
            ascii_chars = [
                ch for ch in non_space_chars
                if 0x20 <= ord(ch) <= 0x7E
            ]
            score = len(ascii_chars) / len(non_space_chars)

        # Keep the best scoring decoding
        if score > best_score:
            best_score = score
            best_text = text

    if best_text is None:
        print(f"WARNING: Could not decode transcript {path}: {last_error}")
        return ""

    # Collapse whitespace/newlines so it plays nicely in CSV cells
    cleaned = " ".join(best_text.split())
    return cleaned



def main():
    # 1. Load metadata
    df = pd.read_csv(METADATA_PATH)

    if "article_id" not in df.columns:
        raise ValueError("metadata CSV must contain an 'article_id' column")
    if "image_display_template" not in df.columns:
        raise ValueError("metadata CSV must contain an 'image_display_template' column")

    # 2. Normalize key columns (strip whitespace, force string)
    df["article_id"] = df["article_id"].astype(str).str.strip()
    df["image_display_template"] = df["image_display_template"].astype(str).str.strip()

    # 3. Ensure transcripts column exists
    if "transcripts" not in df.columns:
        df["transcripts"] = ""
    else:
        # fill NaN with empty string
        df["transcripts"] = df["transcripts"].fillna("")

    # 4. Work only with article rows (compound_object) – robust to whitespace
    mask_articles = df["image_display_template"].eq("compound_object")
    article_rows = df[mask_articles]

    print(f"Found {len(article_rows)} rows with image_display_template == 'compound_object'")

    article_ids = article_rows["article_id"].dropna().unique()

    transcript_by_article = {}

    for article_id in article_ids:
        article_id_str = str(article_id).strip()

        # objects/AlbanyEveningJournal1853/AlbanyEveningJournal1853_InitTransc.txt
        transcript_path = os.path.join(
            OBJECTS_DIR,
            article_id_str,
            f"{article_id_str}_InitTransc.txt"
        )

        print(f"Processing article_id={article_id_str} -> {transcript_path}")
        transcript_text = read_transcript_file(transcript_path)
        transcript_by_article[article_id_str] = transcript_text

    # 5. Fill transcripts ONLY for compound_object rows
    def assign_transcript(row):
        if row["image_display_template"].strip() != "compound_object":
            return row.get("transcripts", "")
        article_id_str = str(row["article_id"]).strip()
        return transcript_by_article.get(article_id_str, "")

    df["transcripts"] = df.apply(assign_transcript, axis=1)

    # 6. Save to CSV
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved updated metadata with transcripts to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


Found 481 rows with image_display_template == 'compound_object'
Processing article_id=AlbanyEveningJournal1853 -> objects/AlbanyEveningJournal1853/AlbanyEveningJournal1853_InitTransc.txt
Processing article_id=AlbanyEveningJournal1879 -> objects/AlbanyEveningJournal1879/AlbanyEveningJournal1879_InitTransc.txt
Processing article_id=AlexandriaGazette1889 -> objects/AlexandriaGazette1889/AlexandriaGazette1889_InitTransc.txt
Processing article_id=AlleganyCountyRepublican1901 -> objects/AlleganyCountyRepublican1901/AlleganyCountyRepublican1901_InitTransc.txt
Processing article_id=Altrocchi1944 -> objects/Altrocchi1944/Altrocchi1944_InitTransc.txt
Processing article_id=AmericanAnthropologist1905 -> objects/AmericanAnthropologist1905/AmericanAnthropologist1905_InitTransc.txt
Processing article_id=ArizonaWeeklyJournalMiner1893 -> objects/ArizonaWeeklyJournalMiner1893/ArizonaWeeklyJournalMiner1893_InitTransc.txt
Processing article_id=ArkansasCityTraveler1892 -> objects/ArkansasCityTraveler1892/A